In [0]:
bootstrap_server = dbutils.secrets.get(scope='fraud-detection', key='bootstrap_server')
topic_name = dbutils.secrets.get(scope='fraud-detection', key='topic_name')
api_key = dbutils.secrets.get(scope='fraud-detection', key='api_key')
api_secret = dbutils.secrets.get(scope='fraud-detection', key='api_secret')

jaas_string = f"""kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_key}" password="{api_secret}";"""

In [0]:
df = spark.readStream.format('kafka')\
    .option("kafka.bootstrap.servers", bootstrap_server)\
    .option("subscribe", topic_name)\
    .option("kafka.security.protocol", "SASL_SSL")\
    .option("kafka.sasl.mechanism", "PLAIN")\
    .option("kafka.sasl.jaas.config", jaas_string)\
    .option("startingOffsets", "earliest")\
    .option("maxOffsetsPerTrigger", "10")\
    .load()

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import StringType
df = df.select(col('key').cast(StringType()).alias('key')\
              , col('value').cast(StringType()).alias('value')
              , col('topic'), col('partition'), col('offset')
              , col('timestamp'))

In [0]:
sQuery = df.writeStream\
            .queryName('kafka-transaction-stream-ingest')\
            .outputMode('append')\
            .option('checkpointLocation', '/Volumes/fraud_detection/source/checkpointlocation/transaction/')\
            .trigger(availableNow=True)\
            .toTable('fraud_detection.bronze.transactions')

In [0]:
%sql
select count(*) from fraud_detection.bronze.transactions